# Model Architecture Comparison

This notebook compares the three NILM model architectures:
- **LSTM**: 3-layer bidirectional LSTM
- **TCN**: 8-layer Temporal Convolutional Network
- **ATCN**: TCN with multi-head self-attention

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from edge_pipeline.models.lstm import LSTMModel
from edge_pipeline.models.tcn import TCNModel
from edge_pipeline.models.atcn import ATCNModel

## Model Parameter Counts

In [ ]:
models = {
    "LSTM (128 hidden, 3 layers)": LSTMModel(window_size=60, hidden_size=128, num_layers=3),
    "TCN (128 channels, 8 layers)": TCNModel(window_size=60, num_channels=128, num_layers=8),
    "ATCN (128 ch, 8 TCN + 2 attn)": ATCNModel(window_size=60, num_channels=128, num_tcn_layers=8),
}

param_counts = {}
for name, model in models.items():
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    param_counts[name] = {"Total": total, "Trainable": trainable}
    print(f"{name}:")
    print(f"  Total parameters: {total:,}")
    print(f"  Trainable parameters: {trainable:,}")
    print()

In [ ]:
# Visualise parameter counts
fig, ax = plt.subplots(figsize=(10, 5))

names = list(param_counts.keys())
totals = [p["Total"] / 1e6 for p in param_counts.values()]

bars = ax.barh(names, totals, color=["#E41A1C", "#377EB8", "#4DAF4A"], edgecolor="black")
ax.set_xlabel("Parameters (millions)")
ax.set_title("Model Parameter Counts")

for bar, total in zip(bars, totals):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
            f"{total:.2f}M", va="center")

plt.tight_layout()
plt.show()

## Inference Speed Comparison

In [ ]:
import time

batch_sizes = [1, 16, 64, 256]
n_iterations = 100

timing_results = {name: {} for name in models.keys()}

for name, model in models.items():
    model.eval()
    for batch_size in batch_sizes:
        x = torch.randn(batch_size, 60)
        
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = model(x)
        
        # Timing
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_iterations):
                _ = model(x)
        elapsed = (time.perf_counter() - start) / n_iterations * 1000  # ms
        
        timing_results[name][batch_size] = elapsed
        print(f"{name} (batch={batch_size}): {elapsed:.2f} ms")

In [ ]:
# Plot inference times
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(batch_sizes))
width = 0.25
colors = ["#E41A1C", "#377EB8", "#4DAF4A"]

for i, (name, times) in enumerate(timing_results.items()):
    vals = [times[bs] for bs in batch_sizes]
    ax.bar(x + i*width, vals, width, label=name.split("(")[0].strip(), color=colors[i], edgecolor="black")

ax.set_xticks(x + width)
ax.set_xticklabels([f"Batch {bs}" for bs in batch_sizes])
ax.set_ylabel("Inference Time (ms)")
ax.set_title("Inference Time by Model and Batch Size")
ax.legend()

plt.tight_layout()
plt.show()

## Receptive Field Analysis

In [ ]:
# TCN receptive field calculation
# Receptive field = (kernel_size - 1) * sum(dilations) + 1

def tcn_receptive_field(kernel_size: int, num_layers: int) -> int:
    dilations = [2**i for i in range(num_layers)]
    return (kernel_size - 1) * sum(dilations) + 1

configs = [
    (3, 4),
    (3, 6),
    (3, 8),
    (5, 4),
    (5, 6),
]

print("TCN Receptive Fields:")
print("-" * 40)
for kernel, layers in configs:
    rf = tcn_receptive_field(kernel, layers)
    print(f"Kernel={kernel}, Layers={layers}: {rf} timesteps")

print("\n(Window size = 60, so RF > 60 means full window coverage)")

## Output Example

In [ ]:
# Generate example output
x = torch.randn(1, 60)

print("Example outputs for random input:")
print("-" * 60)

appliances = ["EVSE", "PV", "CS", "CHP", "BA"]

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        output = model(x).numpy()[0]
    
    print(f"\n{name.split('(')[0].strip()}:")
    for app, val in zip(appliances, output):
        print(f"  {app}: {val:.4f}")